In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a program for multi-head self-attention. Given three input matrices $Q$ (queries), $K$ (keys), and $V$ (values) of size $N \times d_{\text{model}}$, compute:
  $$ \text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1,\ldots,\text{head}_h) $$
  where each head computes:
  $$ \text{head}_i = \text{softmax}\left(\frac{Q_iK_i^T}{\sqrt{d_k}}\right)V_i $$
  with $d_k = d_{\text{model}}/h$ and $Q_i, K_i, V_i$ being the i-th head's partition of the input matrices.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>output</code> array</li>
</ul>

<h2>Example 1:</h2>
<p>
Input:
$$
\begin{align*}
N &= 2, \quad d_{\text{model}} = 4, \quad h = 2 \$$1em]
Q &= \begin{bmatrix}
1.0 & 0.0 & 2.0 & 3.0 \\
4.0 & 5.0 & 6.0 & 7.0
\end{bmatrix} \$$1em]
K &= \begin{bmatrix}
1.0 & 2.0 & 3.0 & 4.0 \\
5.0 & 6.0 & 7.0 & 8.0
\end{bmatrix} \$$1em]
V &= \begin{bmatrix}
0.5 & 1.0 & 1.5 & 2.0 \\
2.5 & 3.0 & 3.5 & 4.0
\end{bmatrix}
\end{align*}
$$

Output:
$$
\begin{bmatrix}
2.39 & 2.89 & 3.50 & 4.00 \\
2.50 & 3.00 & 3.50 & 4.00
\end{bmatrix}
$$
</p>

<h2>Example 2:</h2>
<p>
Input:
$$
\begin{align*}
N &= 1, \quad d_{\text{model}} = 2, \quad h = 1 \$$1em]
Q &= \begin{bmatrix} 1.0 & 1.0 \end{bmatrix} \$$1em]
K &= \begin{bmatrix} 1.0 & 1.0 \end{bmatrix} \$$1em]
V &= \begin{bmatrix} 2.0 & 3.0 \end{bmatrix}
\end{align*}
$$

Output:
$$
\begin{bmatrix} 2.0 & 3.0 \end{bmatrix}
$$
</p>

<h2>Constraints</h2>
<ul>
  <li><code>1 ≤ N ≤ 10000</code></li>
  <li><code>2 ≤ d_model ≤ 1024</code></li>
  <li><code>1 ≤ h ≤ d_model</code></li>
  <li><code>d_model % h == 0</code></li>
  <li><code>-10.0 ≤ values ≤ 10.0</code></li>

  <li>Performance is measured with <code>N</code> = 1,024, <code>d_model</code> = 1,024</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// Q, K, V, output are device pointers
extern "C" void solve(const float* Q, const float* K, const float* V, float* output, int N,
                      int d_model, int h) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# Q, K, V, output are tensors on the GPU
@cute.jit
def solve(
    Q: cute.Tensor,
    K: cute.Tensor,
    V: cute.Tensor,
    output: cute.Tensor,
    N: cute.Int32,
    d_model: cute.Int32,
    h: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# Q, K, V are tensors on the GPU
@jax.jit
def solve(Q: jax.Array, K: jax.Array, V: jax.Array, N: int, d_model: int, h: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    Q: UnsafePointer[Float32, MutExternalOrigin],
    K: UnsafePointer[Float32, MutExternalOrigin],
    V: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
    d_model: Int32,
    h: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# Q, K, V, output are tensors on the GPU
def solve(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    output: torch.Tensor,
    N: int,
    d_model: int,
    h: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# Q, K, V, output are tensors on the GPU
def solve(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    output: torch.Tensor,
    N: int,
    d_model: int,
    h: int,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Multi-Head Attention", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self,
        Q: torch.Tensor,
        K: torch.Tensor,
        V: torch.Tensor,
        output: torch.Tensor,
        N: int,
        d_model: int,
        h: int,
    ):
        assert Q.shape == (N, d_model)
        assert K.shape == (N, d_model)
        assert V.shape == (N, d_model)
        assert output.shape == (N, d_model)
        assert Q.dtype == K.dtype == V.dtype == output.dtype
        assert Q.device == K.device == V.device == output.device
        d_k = d_model // h
        result = torch.zeros((N, d_model), dtype=Q.dtype, device=Q.device)
        for head in range(h):
            Q_h = Q[:, head * d_k : (head + 1) * d_k]
            K_h = K[:, head * d_k : (head + 1) * d_k]
            V_h = V[:, head * d_k : (head + 1) * d_k]
            scores = torch.matmul(Q_h, K_h.t()) / (d_k**0.5)
            softmax = torch.softmax(scores, dim=1)
            head_output = torch.matmul(softmax, V_h)
            result[:, head * d_k : (head + 1) * d_k] = head_output
        output.copy_(result)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "Q": (ctypes.POINTER(ctypes.c_float), "in"),
            "K": (ctypes.POINTER(ctypes.c_float), "in"),
            "V": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "N": (ctypes.c_int, "in"),
            "d_model": (ctypes.c_int, "in"),
            "h": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        Q = torch.tensor([[1.0, 0.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0]], device="cuda", dtype=dtype)
        K = torch.tensor([[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]], device="cuda", dtype=dtype)
        V = torch.tensor([[0.5, 1.0, 1.5, 2.0], [2.5, 3.0, 3.5, 4.0]], device="cuda", dtype=dtype)
        output = torch.empty(2, 4, device="cuda", dtype=dtype)
        return {
            "Q": Q,
            "K": K,
            "V": V,
            "output": output,
            "N": 2,
            "d_model": 4,
            "h": 2,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        test_cases = []
        # basic_example
        Q = torch.tensor([[1.0, 0.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0]], device="cuda", dtype=dtype)
        K = torch.tensor([[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]], device="cuda", dtype=dtype)
        V = torch.tensor([[0.5, 1.0, 1.5, 2.0], [2.5, 3.0, 3.5, 4.0]], device="cuda", dtype=dtype)
        output = torch.empty(2, 4, device="cuda", dtype=dtype)
        test_cases.append({"Q": Q, "K": K, "V": V, "output": output, "N": 2, "d_model": 4, "h": 2})
        # single_head
        Q = torch.tensor([[1.0, 1.0], [2.0, 2.0]], device="cuda", dtype=dtype)
        K = torch.tensor([[1.0, 1.0], [1.0, 1.0]], device="cuda", dtype=dtype)
        V = torch.tensor([[2.0, 3.0], [4.0, 5.0]], device="cuda", dtype=dtype)
        output = torch.empty(2, 2, device="cuda", dtype=dtype)
        test_cases.append({"Q": Q, "K": K, "V": V, "output": output, "N": 2, "d_model": 2, "h": 1})
        # four_heads (random)
        Q = torch.empty(4, 4, device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        K = torch.empty(4, 4, device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        V = torch.empty(4, 4, device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        output = torch.empty(4, 4, device="cuda", dtype=dtype)
        test_cases.append({"Q": Q, "K": K, "V": V, "output": output, "N": 4, "d_model": 4, "h": 4})
        # medium_size (random)
        Q = torch.empty(32, 32, device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        K = torch.empty(32, 32, device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        V = torch.empty(32, 32, device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        output = torch.empty(32, 32, device="cuda", dtype=dtype)
        test_cases.append(
            {"Q": Q, "K": K, "V": V, "output": output, "N": 32, "d_model": 32, "h": 8}
        )
        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        Q = torch.empty(1024, 1024, device="cuda", dtype=dtype).uniform_(-10.0, 10.0)
        K = torch.empty(1024, 1024, device="cuda", dtype=dtype).uniform_(-10.0, 10.0)
        V = torch.empty(1024, 1024, device="cuda", dtype=dtype).uniform_(-10.0, 10.0)
        output = torch.zeros(1024, 1024, device="cuda", dtype=dtype)
        return {
            "Q": Q,
            "K": K,
            "V": V,
            "output": output,
            "N": 1024,
            "d_model": 1024,
            "h": 16,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
